# Likely spurious recombinants with many descendants

A handful of recombination nodes in the sc2ts ARG have very many descendant samples. This
notebook describes those that are not well supported, for the supplementary section "Likely
spurious recombinants with many descendants". The XBB origin also has many descendants, but is
a well supported designated recombinant and is not considered here, and the multi-parent
BA.2.86.1 origin is described in `analysis_multi_parent_recombinants.ipynb`.

In [1]:
import json
import collections

import numpy as np
import pandas as pd
import tszip
import sc2ts

In [2]:
ts = tszip.load("../data/sc2ts_viridian_v2.3.trees.tsz")
df_node = sc2ts.node_data(ts, inheritance_stats=False)
dfr = pd.read_csv("../data/recombinants.csv").set_index("recombinant")
dfr["mutations_averted"] = dfr["k1000_muts"] - dfr["num_mutations"]

# Corroboration by 3SEQ (by recombinant node) and CovRecomb (by sample), as in
# fig_recombinant_evidence.ipynb
df_3seq = pd.read_csv("../arg_postprocessing/sc2ts_v2_2024-06-06_pr_pp_mp_aph_bps_pango_dated_3seq.csv")
dfr["is_3seq_recombinant"] = dfr.index.isin(df_3seq["recombinant"])
df_cr = pd.read_csv("../arg_postprocessing/sc2ts_v2_2024-06-06_pr_pp_mp_aph_bps_pango_dated_CovRecomb_output.csv")
dfr["is_cov_recomb_recombinant"] = dfr["sample_id"].isin(df_cr["sample_id"])

df_pe = pd.read_csv("../data/pango_events_in_arg.csv")
with open("../arg_postprocessing/sc2ts_v2_2024-06-06_pr_pp_mp_aph_bps_pango_dated_k1000_muts.json") as f:
    rematch_muts = {int(k): v for k, v in json.load(f).items()}

## Recombination nodes with the most descendants

Recombinants in `recombinants.csv` (two parents) by number of descendant samples, with the
multi-parent recombinants (not in `recombinants.csv`) added for comparison.

In [3]:
multi_parent = sorted(set(rematch_muts) - set(dfr.index))
num_descendants = np.zeros(len(multi_parent), dtype=int)
for tree in ts.trees():
    num_descendants = np.maximum(num_descendants, [tree.num_samples(u) for u in multi_parent])
df_desc = pd.concat([
    dfr[["recombinant_pango", "num_descendant_samples"]].assign(num_parents=2),
    pd.DataFrame({
        "recombinant_pango": df_node.loc[multi_parent, "pango"].values,
        "num_descendant_samples": num_descendants,
        "num_parents": [np.unique(ts.edges_parent[ts.edges_child == u]).shape[0] for u in multi_parent],
    }, index=multi_parent),
]).sort_values("num_descendant_samples", ascending=False)
df_desc.head(8)

,recombinant_pango,num_descendant_samples,num_parents
1428731,BA.5,244601,2
1213532,BA.2,244192,2
1673452,XBB,112220,2
1854754,BA.2.86.1,25701,3
1073563,BA.2,1330,2
1705726,XBF,832,2
242319,B.1,426,3
343850,AY.5,410,2


In [4]:
xbb_root = 1673452
ba2861_root = 1854754
large = df_desc[df_desc["num_descendant_samples"] > 10_000].index
assert set(large) == {1428731, 1213532, xbb_root, ba2861_root}
# The next largest has fewer than 2000 descendants
assert df_desc["num_descendant_samples"].iloc[4] < 2000
focal = [u for u in large if u not in (xbb_root, ba2861_root)]
focal

[1428731, 1213532]

## Summary

In [5]:
origins = df_pe.set_index("root")["pango"]
df_focal = dfr.loc[focal, [
    "recombinant_pango", "recombinant_date", "num_descendant_samples",
    "interval_left", "interval_right", "parent_left_pango", "parent_right_pango",
    "num_mutations", "k1000_muts", "mutations_averted",
    "net_min_supporting_loci_lft", "net_min_supporting_loci_rgt",
    "net_min_supporting_loci_lft_rgt_ge_4",
    "is_3seq_recombinant", "is_rebar_recombinant", "is_cov_recomb_recombinant",
]].copy()
df_focal["pango_origin"] = [origins.get(u, "") for u in focal]
df_focal.T

recombinant,1428731,1213532
recombinant_pango,BA.5,BA.2
recombinant_date,2022-03-01,2021-12-24
num_descendant_samples,244601,244192
interval_left,23041,26859
interval_right,26858,27382
parent_left_pango,BA.4,BA.2.10
parent_right_pango,BA.2,BA.1.1
num_mutations,4,0
k1000_muts,9,5
mutations_averted,5,5


`num_mutations` is the number of mutations needed with recombination (k=4) and `k1000_muts`
without; `net_min_supporting_loci_*` are the net numbers of supporting loci either side of the
breakpoint used for QC (both must be at least 4).

## Descendants by Pango lineage

In [6]:
def best_tree(u):
    # The tree in which u has the most descendant samples
    index = np.argmax([tree.num_samples(u) for tree in ts.trees()])
    return ts.at_index(index)

descendants = {}
for u in focal:
    descendants[u] = set(best_tree(u).samples(u))
    counts = collections.Counter(df_node.loc[list(descendants[u]), "pango"])
    print(u, df_node.loc[u, "pango"], len(descendants[u]), counts.most_common(8))

1428731 BA.5 244601 [('BA.5.2.1', 39562), ('BA.5.1', 28431), ('BA.5.2', 20996), ('BQ.1.1', 13801), ('BA.5.5', 10926), ('BA.5.6', 7281), ('BE.1.1', 5212), ('BF.7', 4212)]


1213532 BA.2 244192 [('BA.5.2.1', 39571), ('BA.5.1', 28465), ('BA.5.2', 20998), ('BQ.1.1', 13801), ('BA.5.5', 10927), ('BA.5.6', 7282), ('BE.1.1', 5212), ('BF.7', 4212)]


## The two events are nested

The BA.2 node (1213532) is ancestral to the BA.5 origin (1428731) to the right of the BA.5
breakpoint, so the two nodes share almost all of their descendants.

In [7]:
ba5, ba2 = 1428731, 1213532
# Edges (parent segments) of each recombination node
for u in [ba5, ba2]:
    edges = np.where(ts.edges_child == u)[0]
    for e in edges[np.argsort(ts.edges_left[edges])]:
        p = ts.edges_parent[e]
        print(u, int(ts.edges_left[e]), int(ts.edges_right[e]), p, df_node.loc[p, "pango"])

1428731 0 26858 871740 BA.4
1428731 26858 29904 1312318 BA.2
1213532 0 27382 1040012 BA.2.10
1213532 27382 29904 1133699 BA.1.1


In [8]:
# Genomic intervals over which the BA.5 origin descends from the BA.2 node
intervals = []
for tree in ts.trees():
    if tree.is_descendant(ba5, ba2):
        if intervals and intervals[-1][1] == tree.interval.left:
            intervals[-1][1] = tree.interval.right
        else:
            intervals.append([tree.interval.left, tree.interval.right])
intervals

[[26858.0, 29904.0]]

In [9]:
shared = descendants[ba5] & descendants[ba2]
len(descendants[ba5]), len(descendants[ba2]), len(shared)

(244601, 244192, 243642)